# Predictive Models

### Regression Analysis 
- **Predictive Academic Impact** - Modeling how sleep and social media hours predict academic performance.

In [1]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler


KeyboardInterrupt: 

### Loading our cleaned data

In [ ]:
from load_cleaned_data import load_cleaned_data

df = load_cleaned_data()

In [ ]:
# Preparing the data
# Select features and target
features = ['sleep_hours', 'daily_social_media_hours']
target = 'academic_performance'

# Convert to numpy (sklearn doesn't work with Polars directly)
X = df.select(features).to_numpy()
y = df.select(target).to_numpy().ravel()

In [ ]:
# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [ ]:
# Evaluate
r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R² Score : {r2:.4f}")
print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print()
for feat, coef in zip(features, model.coef_):
    print(f"  {feat:30s} → coefficient: {coef:.4f}")
print(f"  {'Intercept':30s} → {model.intercept_:.4f}")

In [ ]:
# Visualize — Actual vs Predicted
fig = px.scatter(
    x=y_test,
    y=y_pred,
    labels={'x': 'Actual Academic Performance', 'y': 'Predicted Academic Performance'},
    title='Actual vs Predicted Academic Performance'
)

# Perfect prediction line
fig.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)

fig.show()

In [ ]:
# Visualize — Feature Coefficients
coef_df = pl.DataFrame({
    'feature': features,
    'coefficient': model.coef_.tolist()
})

fig2 = px.bar(
    coef_df,
    x='feature',
    y='coefficient',
    color='coefficient',
    color_continuous_scale='RdYlGn',
    title='Feature Coefficients — Impact on Academic Performance'
)
fig2.show()

In [ ]:
# Visualize — Residuals
residuals = y_test - y_pred

fig3 = px.scatter(
    x=y_pred,
    y=residuals,
    labels={'x': 'Predicted Values', 'y': 'Residuals'},
    title='Residual Plot'
)

# Zero residual line
fig3.add_hline(y=0, line_dash='dash', line_color='red')
fig3.show()

Interpreting Results
MetricMeaningR²% of variance explained (closer to 1 = better)RMSEAverage prediction error in original units+ve coefficientFeature increases academic performance-ve coefficientFeature decreases academic performance

You can extend this further by adding more features like stress_level or addiction_level to the features list — just add them and rerun. 